This notebook explores variability in hail's python (macro)-benchmarks when
said benchmarks are executed on the hail batch service. The analyses within 
are based off the methods proposed in [1], albeit slightly modified for long
running benchmarks. The goals of these analyses are

- to determine if we can detect slowdowns of 5% or less reliably when running
  benchmarks on hail batch.
- to identify configurations (number of batch jobs x iterations) that allow us
  to detect slowdowns efficiently (ie without excesssive time and money).

[1] Laaber et al., Software Microbenchmarking in the Cloud.How Bad is it Really?
    https://dl.acm.org/doi/10.1007/s10664-019-09681-1

In [ ]:
from pathlib import Path
from typing import Dict, List

import plotly.io

from benchmark.tools import maybe
from benchmark.tools.impex import dump_tsv, import_timings
from benchmark.tools.plotting import plot_mean_time_per_instance, plot_trial_against_time
from benchmark.tools.statistics import (
    bootstrap_mean_confidence_interval,
    laaber_mds,
    schultz_mds,
    variability,
)
from IPython.display import clear_output
from plotly.io import renderers

import hail as hl

In [ ]:
prefix = str(Path().absolute())
hl.init(backend='spark', idempotent=True)

In [ ]:
# Import benchmark data
# ---------------------
#
# benchmarks under `hail/python/benchmarks` are executed with a custom pytest
# plugin and their results are output as json lines (.jsonl).
# Unscrupulously, we use hail to analyse itself.

with hl.TemporaryFilename(dir='/tmp') as tsvfile:
    timings = Path(tsvfile)
    dump_tsv(Path(f'{prefix}/in/benchmarks.jsonl'), timings)
    ht = import_timings(timings)
    ht = ht.checkpoint(f'{prefix}/out/imported.ht', overwrite=True)

In [ ]:
ht = hl.read_table(f'{prefix}/out/imported.ht')
benchmarks = ht.aggregate(hl.agg.collect_as_set(ht.path + hl.str('::') + ht.name))
benchmarks = sorted(benchmarks)
print(*benchmarks, sep='\n')

In [ ]:
# In this next section, we'll estimate the number of iterations required for
# a benchmark to reach a steady-state, or the number of so-called "burn-in"
# iterations.

first_stable_index = {
    'benchmark_analyze_benchmarks': 2,
    'benchmark_block_matrix_to_matrix_table_row_major': 4,
    'benchmark_blockmatrix_write_from_entry_expr_range_mt_standardize': 8,
    'benchmark_blockmatrix_write_from_entry_expr_range_mt': 10,
    'benchmark_concordance': 2,
    'benchmark_export_range_matrix_table_col_p100': 15,
    'benchmark_export_range_matrix_table_entry_field_p100': 3,
    'benchmark_export_range_matrix_table_row_p100': 8,
    'benchmark_export_vcf': 15,
    'benchmark_genetics_pipeline': 4,
    'benchmark_gnomad_coverage_stats_optimized': 5,
    'benchmark_gnomad_coverage_stats': 5,
    'benchmark_group_by_collect_per_row': 5,
    'benchmark_group_by_take_rekey': 4,
    'benchmark_hwe_normalized_pca_blanczos_small_data_0_iterations': 2,
    'benchmark_hwe_normalized_pca_blanczos_small_data_10_iterations': 4,
    'benchmark_hwe_normalized_pca': 6,
    'benchmark_import_and_transform_gvcf': 2,
    'benchmark_import_bgen_filter_count': 18,
    'benchmark_import_bgen_force_count_all': 20,
    'benchmark_import_bgen_force_count_just_gp': 20,
    'benchmark_import_bgen_info_score': 12,
    'benchmark_import_gvcf_force_count': 2,
    'benchmark_import_vcf_count_rows': 1,
    'benchmark_import_vcf_write': 5,
    'benchmark_join_partitions_table[10-10]': 4,
    'benchmark_join_partitions_table[10-100]': 2,
    'benchmark_join_partitions_table[10-1000]': 5,
    'benchmark_join_partitions_table[100-10]': 10,
    'benchmark_join_partitions_table[100-100]': 10,
    'benchmark_join_partitions_table[100-1000]': 8,
    'benchmark_join_partitions_table[1000-10]': 12,
    'benchmark_join_partitions_table[1000-100]': 10,
    'benchmark_join_partitions_table[1000-1000]': 8,
    'benchmark_kyle_sex_specific_qc': 3,
    'benchmark_large_range_matrix_table_sum': 5,
    'benchmark_ld_prune_profile_25': 10,
    'benchmark_linear_regression_rows': 10,
    'benchmark_logistic_regression_rows_wald': 5,
    'benchmark_make_ndarray': 5,
    'benchmark_matrix_table_aggregate_entries': 8,
    'benchmark_matrix_table_array_arithmetic': 20,
    'benchmark_matrix_table_call_stats_star_star': 8,
    'benchmark_matrix_table_cols_show': 5,
    'benchmark_matrix_table_decode_and_count_just_gt': 5,
    'benchmark_matrix_table_decode_and_count': 8,
    'benchmark_matrix_table_entries_show': 4,
    'benchmark_matrix_table_entries_table_no_key': 4,
    'benchmark_matrix_table_entries_table': 10,
    'benchmark_matrix_table_filter_entries_unfilter': 3,
    'benchmark_matrix_table_filter_entries': 6,
    'benchmark_matrix_table_many_aggs_col_wise': 3,
    'benchmark_matrix_table_many_aggs_row_wise': 2,
    'benchmark_matrix_table_rows_force_count': 30,
    'benchmark_matrix_table_rows_is_transition': 5,
    'benchmark_matrix_table_rows_show': 10,
    'benchmark_matrix_table_scan_count_rows_2': 5,
    'benchmark_matrix_table_show': 7,
    'benchmark_matrix_table_take_col': 10,
    'benchmark_matrix_table_take_entry': 8,
    'benchmark_matrix_table_take_row': 10,
    'benchmark_mt_group_by_memory_usage': 5,
    'benchmark_mt_localize_and_collect': 5,
    'benchmark_ndarray_addition': 10,
    'benchmark_ndarray_matmul_float64': 6,
    'benchmark_ndarray_matmul_int64': 2,
    'benchmark_pc_relate_5k_5k': 4,
    'benchmark_pc_relate': 3,
    'benchmark_per_row_stats_star_star': 10,
    'benchmark_python_only_10k_combine': 20,
    'benchmark_python_only_10k_transform': 10,
    'benchmark_read_decode_gnomad_coverage': 10,
    'benchmark_read_force_count_partitions[10]': 10,
    'benchmark_read_force_count_partitions[100]': 8,
    'benchmark_read_force_count_partitions[1000]': 10,
    'benchmark_read_with_index[1000]': 3,
    'benchmark_sample_qc': 3,
    'benchmark_sentinel_cpu_hash_1': 5,
    'benchmark_sentinel_read_gunzip': 10,
    'benchmark_shuffle_key_by_aggregate_bad_locality': 8,
    'benchmark_shuffle_key_by_aggregate_good_locality': 5,
    'benchmark_shuffle_key_rows_by_4096_byte_rows': 2,
    'benchmark_shuffle_key_rows_by_65k_byte_rows': 4,
    'benchmark_shuffle_key_rows_by_mt': 10,
    'benchmark_shuffle_order_by_10m_int': 22,
    'benchmark_split_multi_hts': 10,
    'benchmark_split_multi': 2,
    'benchmark_sum_table_of_ndarrays': 10,
    'benchmark_table_aggregate_approx_cdf': 16,
    'benchmark_table_aggregate_array_sum': 5,
    'benchmark_table_aggregate_counter': 20,
    'benchmark_table_aggregate_downsample_dense': 5,
    'benchmark_table_aggregate_downsample_worst_case': 10,
    'benchmark_table_aggregate_int_stats': 5,
    'benchmark_table_aggregate_linreg': 8,
    'benchmark_table_aggregate_take_by_strings': 10,
    'benchmark_table_annotate_many_flat': 5,
    'benchmark_table_big_aggregate_compilation': 4,
    'benchmark_table_big_aggregate_compile_and_execute': 2,
    'benchmark_table_expr_take': 20,
    'benchmark_table_foreign_key_join[1000000-1000]': 3,
    'benchmark_table_foreign_key_join[1000000-1000000]': 3,
    'benchmark_table_group_by_aggregate_sorted': 8,
    'benchmark_table_group_by_aggregate_unsorted': 7,
    'benchmark_table_import_ints_impute': 8,
    'benchmark_table_import_ints': 3,
    'benchmark_table_import_strings': 3,
    'benchmark_table_key_by_shuffle': 3,
    'benchmark_table_python_construction': 10,
    'benchmark_table_range_array_range_force_count': 6,
    'benchmark_table_range_force_count': 10,
    'benchmark_table_range_join[1000000000-1000]': 20,
    'benchmark_table_range_join[1000000000-1000000000]': 10,
    'benchmark_table_range_means': 10,
    'benchmark_table_read_force_count_ints': 5,
    'benchmark_table_read_force_count_strings': 10,
    'benchmark_table_scan_prev_non_null': 20,
    'benchmark_table_scan_sum_1k_partitions': 2,
    'benchmark_table_show': 20,
    'benchmark_table_take': 20,
    'benchmark_test_head_and_tail_region_memory': 10,
    'benchmark_test_inner_join_region_memory': 10,
    'benchmark_test_left_join_region_memory': 10,
    'benchmark_test_map_filter_region_memory': 10,
    'benchmark_union_partitions_table[10-10]': 3,
    'benchmark_union_partitions_table[10-100]': 5,
    'benchmark_union_partitions_table[10-1000]': 8,
    'benchmark_union_partitions_table[100-10]': 16,
    'benchmark_union_partitions_table[100-100]': 5,
    'benchmark_union_partitions_table[100-1000]': 15,
    'benchmark_union_partitions_table[1000-10]': 4,
    'benchmark_union_partitions_table[1000-100]': 6,
    'benchmark_union_partitions_table[1000-1000]': 8,
    'benchmark_variant_and_sample_qc_nested_with_filters_2': 2,
    'benchmark_variant_and_sample_qc_nested_with_filters_4_counts': 20,
    'benchmark_variant_and_sample_qc_nested_with_filters_4': 10,
    'benchmark_variant_and_sample_qc': 10,
    'benchmark_variant_qc': 5,
    'benchmark_vds_combiner_chr22': 10,
    'benchmark_write_profile_mt': 15,
    'benchmark_write_range_matrix_table_p100': 2,
    'benchmark_write_range_table[10000000-10]': 3,
    'benchmark_write_range_table[10000000-100]': 3,
    'benchmark_write_range_table[10000000-1000]': 6,
    'benchmark_matrix_table_scan_count_cols_2': 20,
}

In [ ]:
# Short of an accurate algorithm for computing this, you, noble reader, are
# tasked with the mind-numbing task of looking at graphs and picking numbers.
# This is an iterative process and you'll likely lose the will to live mid-way.
#
# Persevere, friend. Your sacrifice will not go unrewarded.
#
# In what follows, you'll be shown two plots. On the top will be the unfiltered
# benchmark times vs iteration for all batch jobs. The plot below will show the
# same benchmark filtered to the number of burn in iterations you selected
# previously.
#
# You'll be promted to enter a new first stable index for each benchmark until
# you arrive at a fixed point. Note that some benchmarks never really reach
# stability. In this case try to pick a value that compromises between cost and
# accuracy (ie if a benchmark is really slow, we don't want to be doing tons
# of burn in iterations, whereas for a fast benchmark we could justify more).
#
# Good luck.

ht = hl.read_table(f'{prefix}/out/imported.ht')
names: List[str] = ht.aggregate(hl.agg.collect_as_set(ht.name))  # type: ignore

while len(names) != 0:
    __new_names, names = names, []
    for fig in plot_trial_against_time(ht, __new_names, first_stable_index):
        clear_output(wait=True)

        name: str = fig.labels.title  # type: ignore
        cur_index = first_stable_index.get(name)

        try:
            fig.show()
        except:
            continue

        new_index = maybe(int, input('Enter the first stable index (or blank keep same)') or None)

        if new_index is not None and new_index != cur_index:
            first_stable_index[name] = new_index
            names.append(name)

first_stable_index

In [ ]:
# As a final step of cleaning, we'll filter out trials that differ by some
# multiplier of the median for each instance

def filter_burn_in_iterations(ht: hl.Table, first_stable_index: Dict[str, int]) -> hl.Table:
    ht = ht.annotate_globals(first_stable_index=first_stable_index)
    return ht.select(
        instances=ht.instances.map(
            lambda instance: instance.annotate(
                trials=hl.filter(
                    lambda t: t.iteration >= ht.first_stable_index.get(ht.name, 0),
                    instance.trials,
                ),
            )
        ),
    )

def filter_outliers(ht: hl.Table, factor: hl.Float64Expression) -> hl.Table:
    return ht.select(
        instances=ht.instances.map(
            lambda instance: instance.annotate(
                trials=hl.bind(
                    lambda median: instance.trials.filter(
                        lambda t: hl.max([t.time, median]) / hl.min([t.time, median]) < factor
                    ),
                    hl.median(instance.trials.map(lambda t: t.time)),
                )
            ),
        ),
    )

def filter_names(ht: hl.Table, names: hl.SetExpression) -> hl.Table:
    return ht.filter(names.contains(ht.name))

def filter_failed_trials(ht: hl.Table) -> hl.Table:
    return ht.annotate(
        instances=ht.instances.map(
            lambda i: i.annotate(
                trials=hl.filter(
                    lambda t: ~t.timed_out | hl.is_missing(t.failure),
                    i.trials,
                ),
            )
        ),
    )

def filter_non_viable_instances(ht: hl.Table, ninstances: hl.Int32Expression, ntrials: hl.Int32Expression) -> hl.Table:
    ht = ht.annotate(
        instances=hl.filter(
            lambda instance: hl.len(instance.trials) >= ntrials,
            ht.instances,
        ),
    )

    return ht.filter(hl.len(ht.instances) >= ninstances)


ht = hl.read_table(f'{prefix}/out/imported.ht')
all: List[str] = ht.aggregate(hl.agg.collect_as_set(ht.name))  # type: ignore

ht = filter_names(ht, hl.set([n for n in all if n in first_stable_index]))
ht = filter_burn_in_iterations(ht, first_stable_index)
ht = filter_failed_trials(ht)
ht = filter_outliers(ht, hl.float64(10))
# ht = filter_non_viable_instances(ht, hl.int(50), hl.int(50))
ht = ht.checkpoint(f'{prefix}/out/filtered.ht', overwrite=True)

benchmarks = ht.aggregate(hl.agg.collect_as_set(ht.name))

print('Filtered:', *(n for n in all if n not in set(benchmarks)), sep='\n')

In [ ]:
# These plots show the mean time per instance. This provides a visual way of
# identifying differences in instance type if there are multiple distinct layers

for f in plot_mean_time_per_instance(ht):
    f.show()

In [ ]:
# laaber et al. section 4

ht = ht.select(instances=ht.instances.trials.time)
variability(ht).show(len(benchmarks))

In [ ]:
# laaber et al. section 5 - boostrapping confidence intervals of the mean

bootstrap_mean_confidence_interval(ht, 1000, 0.95).show()

In [ ]:
# Optionally, use QoB for the next section
hl.stop()
hl.init(backend='batch')

new_prefix = hl.current_backend().remote_tmpdir
hl.current_backend().fs.copy(f'{prefix}/out/filtered.ht', f'{new_prefix}/out/filtered.ht')
prefix, new_prefix = new_prefix, prefix

In [ ]:
# Laaber et al - Minimal-Detectable Slowdown

ht = hl.read_table(f'{prefix}/out/filtered.ht')
ht = ht.select(instances=ht.instances.trials.time)

laaber_mds(ht).write(f'{prefix}/out/laaber-mds.ht', overwrite=True)
schultz_mds(ht).write(f'{prefix}/out/schultz-mds.ht', overwrite=True)

In [ ]:
# Switch back to local spark for plotting
for mds in ['laaber', 'schultz']:
    hl.current_backend().fs.copy(f'{prefix}/out/{mds}-mds.ht', f'{new_prefix}/out/{mds}-mds.ht')

prefix, new_prefix = new_prefix, prefix
hl.stop()
hl.init(backend='spark')

In [ ]:
benchmarks = hl.read_table(f'{prefix}/out/filtered.ht')
benchmarks = benchmarks.aggregate(hl.agg.collect_as_set((benchmarks.path, benchmarks.name)))
benchmarks = sorted(benchmarks)

mds, schultz = [hl.read_table(f'{prefix}/out/{m}-mds.ht') for m in ('laaber', 'schultz')]
mds = mds.annotate_globals(first_stable_index=first_stable_index)
mds = mds.select(nburn_in=mds.first_stable_index[mds.name], laaber=mds.row_value, schultz=schultz[mds.key])

for path, name in benchmarks:
    clear_output(wait=True)
    print(f'{path}::{name}')

    t = mds.filter((mds.path == path) & (mds.name == name))
    t.filter(t.slowdown == 1).show(100_000)

    # Select minimum `(instances, trails)` that reduces the rate of false positives
    # to some acceptable value.
    config = input('instances, trials')
    if not config:
        continue

    clear_output(wait=True)
    print(f'{path}::{name}')

    # Now identify a configuration of instances, trials and burn-in iterations
    m, n = [int(x.strip()) for x in config.split(',')]
    t.filter((t.slowdown > 1) & (hl.tuple([t.ninstances, t.ntrials]) >= (m, n))).show(100_000)
    input('Press any key to continue')